[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FutoshiNakamura-Tts/gaussian-splatting-colab/blob/colab-t4-2025-12-18/gaussian_splatting_colab.ipynb)

In [ ]:
# @title 1. Key Imports & Utilities
import os
import sys
import shutil
import subprocess
import time
import re
import shlex
import torch
import glob
from google.colab import drive, files
from threading import Timer
from queue import Queue
from random import randint
import ipywidgets as widgets
from IPython.display import display, clear_output, Javascript

# Output Area for Logs
out = widgets.Output(layout={'border': '1px solid #ddd', 'height': '300px', 'overflow_y': 'scroll'})

def log(msg):
    # Use append_stdout for thread safety with ipywidgets.Output
    out.append_stdout(str(msg) + '\n')

def run_command(cmd, shell=True, env=None):
    try:
        process = subprocess.Popen(
            cmd, shell=shell, 
            stdout=subprocess.PIPE, 
            stderr=subprocess.STDOUT, 
            text=True,
            env=env
        )
        for line in process.stdout:
            log(line.strip())
        process.wait()
        if process.returncode != 0:
             log(f"Command failed with return code {process.returncode}")
    except Exception as e:
        log(f"Error executing command: {e}")


In [ ]:
# @title 2. Configuration
# ===========================
# Configuration Parameters
# ===========================
# @markdown ### Global Config
# (Updated: 2025-12-23 10:07:25)
Mount_Drive = False # @param {type:"boolean"}
Use_Tailscale = False # @param {type:"boolean"}

# @markdown ### Data Config
DataSource = "Demo Data" # @param ["Demo Data", "Google Drive", "Upload Zip", "Custom URL", "Local Folder"]
DrivePath = "/content/drive/MyDrive/my_data.zip" # @param {type:"string"}
Custom_URL = "" # @param {type:"string"}

Repo_Branch = "" # @param {type:"string"}

# @markdown ### Training Config
Source_Path = "/content/tandt/truck" # @param {type:"string"}
Output_Path = "/content/gaussian-splatting/output/my-experiment" # @param {type:"string"}
Iterations = 30000 # @param {type:"integer"}
SH_Degree = 3 # @param {type:"integer"}
White_Background = False # @param {type:"boolean"}
Eval_Mode = False # @param {type:"boolean"}
DryRun = False # @param {type:"boolean"}


In [ ]:
# @title 3. Define Actions
# ===========================
# Action Functions
# ===========================

def setup_env(b):
    out.clear_output()
    log("=== 1. Setting up Environment ===")
    
    if Mount_Drive:
        if not os.path.exists('/content/drive'):
            log("Mounting Google Drive...")
            drive.mount('/content/drive')
        else:
            log("Drive already mounted.")
    
    log("Checking GPU...")
    run_command("nvidia-smi")
    
    log("Environment setup complete.")


def install_deps(b):
    out.clear_output()
    log("=== 2. Installing Dependencies ===")
    os.chdir('/content')
    
    # Check Rasterizer
    rasterizer_branch = 'main'
    is_accelerated = False 
    if 'dd_rasterizer' in globals():
        if dd_rasterizer.value.startswith('Accelerated'):
            rasterizer_branch = '3dgs_accel'
            is_accelerated = True
            log("Selected Accelerated Rasterizer (Sparse Adam).")
        else:
            log("Selected Standard Rasterizer.")

    if not os.path.exists('/content/wheels_repo'):
         log("Cloning wheels repo...")
         run_command("git clone -b colab-t4-2025-12-18 https://github.com/FutoshiNakamura-Tts/gaussian-splatting-colab /content/wheels_repo")
    
    repo_dir = '/content/gaussian-splatting'
    if not os.path.exists(repo_dir):
         log(f"Cloning gaussian-splatting...")
         run_command(f"git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting {repo_dir}")
    else:
         pass
    
    log("Installing plyfile...")
    run_command("pip install -q plyfile")

    # Handle Submodules (Diff-Gaussian-Rasterization)
    dgr_dir = f"{repo_dir}/submodules/diff-gaussian-rasterization"
    sknn_dir = f"{repo_dir}/submodules/simple-knn"
    
    try:
        current_dgr_branch = "unknown"
        log(f"Configuring diff-gaussian-rasterization to branch: {rasterizer_branch}")
        
        # Unconditionally fetch and checkout desired branch for submodule
        run_command(f"git -C {dgr_dir} fetch origin {rasterizer_branch}")
        run_command(f"git -C {dgr_dir} checkout {rasterizer_branch}")
    except Exception as e:
        log(f"Error configuring submodule: {e}")

    log("Checking for bundled wheels or building submodules...")
    
    packages = [
        {"name": "diff-gaussian-rasterization", "pattern": "diff_gaussian_rasterization-*-cp*-*-linux_x86_64.whl", "src": dgr_dir},
        {"name": "simple-knn", "pattern": "simple_knn-*-cp*-*-linux_x86_64.whl", "src": sknn_dir}
    ]
    
    for pkg in packages:
        wheel_to_install = None
        
        # 1. If Accelerated, check subfolder first
        if is_accelerated:
            accel_wheels = glob.glob(f"/content/wheels_repo/wheels/3dgs_accel/{pkg['pattern']}")
            if accel_wheels:
                wheel_to_install = accel_wheels[0]
                log(f"Found Accelerated wheel for {pkg['name']}: {wheel_to_install}")
        
        # 2. Check root wheels folder (Fallback or Standard)
        if not wheel_to_install:
             # Only allow fallback if NOT (Accelerated Mode AND diff-gaussian-rasterization)
             # simple-knn is safe to fallback as it is shared
             allow_fallback = not (is_accelerated and pkg['name'] == 'diff-gaussian-rasterization')
             
             if allow_fallback:
                 standard_wheels = glob.glob(f"/content/wheels_repo/wheels/{pkg['pattern']}")
                 if standard_wheels:
                     wheel_to_install = standard_wheels[0]
                     log(f"Found Standard/Shared wheel for {pkg['name']}: {wheel_to_install}")
             elif is_accelerated:
                 log(f"Skipping standard wheel fallback for {pkg['name']} to force source build from 3dgs_accel branch.")

        if wheel_to_install:
            run_command(f"{sys.executable} -m pip install --force-reinstall -q {wheel_to_install}")
        else:
            log(f"No wheel found for {pkg['name']}. Building from source...")
            run_command(f"{sys.executable} -m pip install --force-reinstall -q {pkg['src']}")

    log("Dependencies installed.")




In [ ]:
def prepare_data(b):
    out.clear_output()
    # Check widgets
    data_source_val = DataSource
    if 'dd_datasource' in globals():
        data_source_val = dd_datasource.value
        
    drive_path_val = DrivePath
    custom_url_val = Custom_URL
    local_folder_val = ""
    
    if 'txt_source_path' in globals():
        # Use the text input for generic path/url
        drive_path_val = txt_source_path.value
        custom_url_val = txt_source_path.value
        local_folder_val = txt_source_path.value

    log(f"=== 3. Preparing Data: {data_source_val} ===")
    os.chdir('/content')
    
    if data_source_val == "Demo Data":
        if not os.path.exists('tandt_db'):
            log("Downloading TandT Demo Data...")
            run_command("wget -q https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandt_db.zip")
            run_command("unzip -q tandt_db.zip")
        else:
            log("Demo data already exists.")

    elif data_source_val == "Google Drive":
        if not os.path.exists(drive_path_val):
            log(f"Error: Drive Path {drive_path_val} does not exist. Did you mount drive in Step 1?")
            return
        
        if os.path.isfile(drive_path_val):
             fname = os.path.basename(drive_path_val)
             if fname.lower().endswith('.zip'):
                 log(f"Copying and unzipping {fname}...")
                 shutil.copy(drive_path_val, f"./{fname}")
                 run_command(f"unzip -q \"{fname}\"")
             else:
                 log(f"Copying {fname}...")
                 shutil.copy(drive_path_val, ".")
        elif os.path.isdir(drive_path_val):
            log(f"Target is a directory: {drive_path_val}. Using it directly via Source_Path is recommended if it is accessible.")

    elif data_source_val == "Upload Zip":
        log("Checking for uploaded file...")
        if 'file_upload_widget' in globals() and file_upload_widget.value:
            # ipywidgets.FileUpload value is a dict: {name: {content: b'', ...}} or tuple/list in newer versions
            # But in Colab usually it's a map. Let's handle generic structure.
            uploaded_data = file_upload_widget.value
            
            # Simple normalization helper
            files_to_process = []
            if isinstance(uploaded_data, dict):
                for name, info in uploaded_data.items():
                    content = info['content']
                    files_to_process.append((name, content))
            elif isinstance(uploaded_data, (list, tuple)):
                for item in uploaded_data:
                    # item is a dict with 'name', 'content' (and 'type', 'size', etc.)
                    name = item['name']
                    content = item['content']
                    # content might be memoryview
                    if isinstance(content, memoryview):
                        content = content.tobytes()
                    files_to_process.append((name, content))
            
            if not files_to_process:
                log("No file uploaded in the widget. Please select a file first.")
                return

            for fname, content in files_to_process:
                log(f"Processing {fname}...")
                with open(fname, 'wb') as f:
                    f.write(content)
                
                if fname.lower().endswith('.zip'):
                    log(f"Unzipping {fname}...")
                    run_command(f"unzip -q \"{fname}\"")
        else:
             log("No file widget found or no file selected. Please upload a file using the 'Upload' button.")
             return
        
    elif data_source_val == "Custom URL":
        if not custom_url_val:
             log("Error: Custom URL is empty.")
             return
        log(f"Downloading from {custom_url_val}...")
        fname = os.path.basename(custom_url_val)
        if '?' in fname: fname = fname.split('?')[0]
        if not fname: fname = "downloaded_data.zip"

        run_command(f"wget -q -O {fname} {custom_url_val}")
        
        if fname.lower().endswith('.zip'):
            log(f"Unzipping {fname}...")
            run_command(f"unzip -q \"{fname}\"")
        else:
            log(f"Downloaded {fname}.")
            
    elif data_source_val == "Local Folder":
        if not local_folder_val:
             log("Error: Local Folder path is empty.")
             return
        if not os.path.exists(local_folder_val):
             log(f"Error: Path {local_folder_val} does not exist.")
             return
        log(f"Using Local Folder: {local_folder_val}")


    # === Validation Step ===
    log("Validating data structure...")
    target_source_path = Source_Path
    if 'txt_source_path' in globals() and txt_source_path.value and data_source_val != "Demo Data":
         pass
         
    if os.path.exists(target_source_path):
        sparse_path = os.path.join(target_source_path, "sparse")
        if not os.path.exists(sparse_path):
            log(f"[WARNING] 'sparse' directory not found in {target_source_path}!")
            log(f"[WARNING] Training requires COLMAP DATA (sparse/0/*.bin).")
            log(f"[WARNING] If you only uploaded images, training WILL FAIL.")
        else:
            log(f"Data looks valid (found 'sparse' directory in {target_source_path}).")
    else:
        log(f"[WARNING] The configured Source_Path '{target_source_path}' does not exist.")

    log("Data preparation complete.")




In [ ]:
def start_training(b):
    out.clear_output()
    log("=== 4. Starting Training ===")
    os.chdir('/content/gaussian-splatting')
    
    dry_run_val = DryRun
    if 'cb_dryrun' in globals():
        dry_run_val = cb_dryrun.value
    
    cmd_parts = [sys.executable, "train.py"]
    cmd_parts.extend(["-s", Source_Path])
    cmd_parts.extend(["-m", Output_Path])
    cmd_parts.extend(["--iterations", str(Iterations)])
    cmd_parts.extend(["--sh_degree", str(SH_Degree)])
    if White_Background: cmd_parts.append("-w")
    if Eval_Mode: cmd_parts.append("--eval")
    
    # New Features Arguments
    if 'cb_antialiasing' in globals() and cb_antialiasing.value:
        cmd_parts.append("--antialiasing")
    
    if 'cb_exposure' in globals() and cb_exposure.value:
        # Defaults from README
        cmd_parts.extend(["--exposure_lr_init", "0.001", "--exposure_lr_final", "0.0001", "--exposure_lr_delay_steps", "5000", "--exposure_lr_delay_mult", "0.001", "--train_test_exp"])
        
    if 'cb_depth' in globals() and cb_depth.value:
        if 'txt_depth_path' in globals() and txt_depth_path.value:
             cmd_parts.extend(["-d", txt_depth_path.value])
        else:
             log("[WARNING] Depth Regularization enabled but no path provided!")

    if 'cb_sparse_adam' in globals() and cb_sparse_adam.value:
        cmd_parts.extend(["--optimizer_type", "sparse_adam"])

    cmd = " ".join(shlex.quote(arg) for arg in cmd_parts)
    
    if dry_run_val or not torch.cuda.is_available():
        log("--- Dry Run Mode or No GPU ---")
        log(f"Command: {cmd}")
    else:
        log(f"Executing: {cmd}")
        run_command(cmd)


def start_viewer(b):
    out.clear_output()
    log("=== 5. Starting Viewer ===")
    
    # Install Cloudflared if needed
    if not os.path.exists('/usr/local/bin/cloudflared') and not os.path.exists('/content/cloudflared-linux-amd64.deb'):
        log("Installing Cloudflared...")
        run_command("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /content/cloudflared-linux-amd64.deb")
        run_command("dpkg -i /content/cloudflared-linux-amd64.deb")
    
    log("Starting Tunnel and File Server...")
    # Start Tunnel in background
    metrics_port = randint(8100, 9000)
    port = 8000
    
    subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}', '--metrics', f'127.0.0.1:{metrics_port}'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    
    log("Waiting for Cloudflare Tunnel URL...")
    import requests
    tunnel_url = None
    for _ in range(20):
        time.sleep(2)
        try:
            resp = requests.get(f'http://127.0.0.1:{metrics_port}/metrics')
            found = re.search(r"(?P<url>https?:\/\/[^\s]+.trycloudflare.com)", resp.text)
            if found:
                tunnel_url = found.group("url")
                break
        except:
            pass
    
    if tunnel_url:
        log(f"\n>>> Files accessible at: {tunnel_url} <<<\n")
        os.environ['webui_url'] = tunnel_url
    else:
        log("Failed to get Tunnel URL.")

    # Start HTTP Server
    serve_dir = Output_Path if os.path.exists(Output_Path) else "/content/gaussian-splatting"
    if not os.path.exists(serve_dir):
        os.makedirs(serve_dir, exist_ok=True)
    
    log(f"Serving {serve_dir} on port {port}...")
    subprocess.Popen([sys.executable, "-m", "http.server", str(port)], cwd=serve_dir)


In [ ]:
# @title 3.5. Tailscale Connection (Class)
# ===========================
# Tailscale Logic
# ===========================

class TailscaleConnection:
    def __init__(self):
        self.connected = False
        self.status_callback = None

    def log(self, msg):
        if self.status_callback:
             # Extract a simple status derived from the message if possible, or just pass 'Busy'
             # For now, we rely on specific status updates in methods
             pass
        # Use global log function from cell 1
        # Use global log function from cell 1
        if 'log' in globals():
            log(msg)
        else:
            print(msg)

    def install(self):
        if self.status_callback: self.status_callback("Installing...")
        if not os.path.exists('/usr/bin/tailscale'):
             self.log("Installing Tailscale...")
             run_command("curl -fsSL https://tailscale.com/install.sh | sh")

    def connect(self):
        if self.status_callback: self.status_callback("Connecting...")
        if self.connected:
            self.log("Tailscale already connected.")
            return
            
        self.log("\n=== Enabling Tailscale ===")
        self.install()
        
        self.log("Configuring Hostname: colab")
        run_command("hostname colab")
        
        self.log("Starting Tailscale Daemon...")
        run_command("nohup tailscaled --tun=userspace-networking --socket=/run/tailscale/tailscaled.sock --port 41641  >/dev/null 2>&1 &")
        
        self.log("Connecting...")
        run_command("tailscale up --ssh --hostname=colab")
        self.log("Tailscale Connected (check output for login link if needed).")
        self.connected = True
        if self.status_callback: self.status_callback("Connected")

    def disconnect(self):
        if not self.connected:
            return
        self.log("\n=== Disabling Tailscale ===")
        run_command("tailscale down")
        self.log("Tailscale Disconnected.")

    def toggle(self, change):
        if change['new']:
            self.connect()
        else:
            self.disconnect()

tailscale_conn = TailscaleConnection()

# Auto-connect if configured globally (e.g. at start)
if 'Use_Tailscale' in globals() and Use_Tailscale:
    tailscale_conn.connect()


In [ ]:
# @title 4.1. GUI Layout (View)
# ===========================
# Defines widgets and layout
# ===========================
import ipywidgets as widgets
from IPython.display import display

class GUIWidgets:
    def __init__(self):
        self.style = {'description_width': 'initial'}
        self.layout = widgets.Layout(width='auto')
        
        # --- Status Indicator ---
        self.status_html = widgets.HTML(
            value='''
            <style>
            .status-box { padding: 5px; border-radius: 4px; font-weight: bold; }
            .status-ready { background-color: #e6ffed; color: #2da44e; border: 1px solid #2da44e; }
            .status-busy { background-color: #fff8c5; color: #bf8b2d; border: 1px solid #bf8b2d; }
            .loader { border: 3px solid #f3f3f3; border-top: 3px solid #3498db; border-radius: 50%; width: 14px; height: 14px; animation: spin 1s linear infinite; display: inline-block; vertical-align: middle; margin-right: 5px; }
            @keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }
            </style>
            <div class="status-box status-ready">✅ Ready</div>
            '''
        )

        # --- Core Widgets ---
        self.lbl_tailscale_status = widgets.Label(value="Disconnected", style={'text_color': 'gray'})
        self.cb_tailscale = widgets.Checkbox(value=False, description='Connect Tailscale (ssh)', style=self.style)
        self.cb_dryrun = widgets.Checkbox(value=False, description='Dry Run (No GPU)', style=self.style)
        
        self.btn_env = widgets.Button(description="1. Setup Environment", button_style='primary', layout=self.layout)
        self.dd_rasterizer = widgets.Dropdown(options=['Standard', 'Accelerated (Sparse Adam)'], value='Standard', description='Rasterizer:', style=self.style)
        self.btn_deps = widgets.Button(description="2. Install Dependencies", button_style='info', layout=self.layout)
        
        self.dd_datasource = widgets.Dropdown(options=['Demo Data', 'Google Drive', 'Upload Zip', 'Custom URL', 'Local Folder'], value='Demo Data', description='Data Source:', style=self.style)
        self.txt_source_path = widgets.Text(value='', placeholder='Drive Path / URL / Folder Path', description='Path / URL:', style=self.style)
        self.btn_browse = widgets.Button(description="📂 Browse", layout=widgets.Layout(width='100px'))
        self.file_upload = widgets.FileUpload(accept='.zip', multiple=False)
        self.file_upload.layout.display = 'none'
        self.lbl_upload_instruction = widgets.HTML("<i>Select 'Upload Zip' and click <b>'3. Prepare Data'</b> to launch the upload.</i>")
        self.lbl_upload_instruction.layout.display = 'none'
        
        # File Browser Widgets
        self.lbl_path = widgets.Label(f"Current: /")
        self.sel_files = widgets.Select(options=[], rows=10, layout=widgets.Layout(width='100%'))
        self.btn_up = widgets.Button(description="⬆ Up", layout=widgets.Layout(width='80px'))
        self.btn_select = widgets.Button(description="Select", button_style='primary', layout=widgets.Layout(width='80px'))
        self.browser_box = widgets.VBox([widgets.HBox([self.btn_up, self.lbl_path]), self.sel_files, self.btn_select])
        self.browser_box.layout.display = 'none'

        self.btn_data = widgets.Button(description="3. Prepare Data", button_style='warning', layout=self.layout)
        
        # Training Options
        self.cb_antialiasing = widgets.Checkbox(value=False, description='Anti-aliasing', style=self.style)
        self.cb_exposure = widgets.Checkbox(value=False, description='Exposure Comp', style=self.style)
        self.cb_depth = widgets.Checkbox(value=False, description='Depth Reg', style=self.style)
        self.txt_depth_path = widgets.Text(value='', placeholder='Depth maps path', description='Depth Path:', display='none', style=self.style)
        self.txt_depth_path.layout.display = 'none'
        self.cb_sparse_adam = widgets.Checkbox(value=False, description='Sparse Adam', disabled=True, style=self.style)
        
        self.btn_train = widgets.Button(description="4. Train", button_style='success', layout=self.layout)
        self.btn_view = widgets.Button(description="5. Start Viewer", button_style='danger', layout=self.layout)
        
        # --- Layout Container ---
        self.container = widgets.VBox([
            widgets.HBox([widgets.HTML("<h3>Gaussian Splatting Controller</h3>"), self.status_html]),
            widgets.HBox([self.cb_tailscale, self.lbl_tailscale_status]),
            widgets.HBox([widgets.HTML("<h3>Gaussian Splatting Controller</h3>"), self.status_html]),
            widgets.HBox([self.btn_env]),
            widgets.HBox([self.dd_rasterizer, self.btn_deps]),
            widgets.HTML("<hr>"),
            widgets.HBox([self.dd_datasource, self.txt_source_path, self.btn_browse]),
            widgets.HBox([self.file_upload, self.lbl_upload_instruction]),
            self.browser_box,
            widgets.HBox([self.btn_data]),
            widgets.HTML("<hr>"),
            widgets.Label("Training Options (New Features):"),
            widgets.HBox([self.cb_antialiasing, self.cb_exposure, self.cb_sparse_adam]),
            widgets.HBox([self.cb_depth, self.txt_depth_path]),
            widgets.HTML("<br>"),
            widgets.HBox([self.cb_dryrun]),
            widgets.HBox([self.btn_train, self.btn_view]),
            widgets.Label("Logs:"),
            out  # Using global 'out' widget
        ])

    def set_status(self, state, msg):
        if state == 'busy':
            self.status_html.value = f'''
            <style>.status-box {{ padding: 5px; border-radius: 4px; font-weight: bold; }} .status-busy {{ background-color: #fff8c5; color: #bf8b2d; border: 1px solid #bf8b2d; }} .loader {{ border: 3px solid #f3f3f3; border-top: 3px solid #3498db; border-radius: 50%; width: 14px; height: 14px; animation: spin 1s linear infinite; display: inline-block; vertical-align: middle; margin-right: 5px; }} @keyframes spin {{ 0% {{ transform: rotate(0deg); }} 100% {{ transform: rotate(360deg); }} }}</style>
            <div class="status-box status-busy"><div class="loader"></div> {msg}</div>'''
        else:
            self.status_html.value = '''
             <style>.status-box {{ padding: 5px; border-radius: 4px; font-weight: bold; }} .status-ready {{ background-color: #e6ffed; color: #2da44e; border: 1px solid #2da44e; }}</style>
            <div class="status-box status-ready">✅ Ready</div>'''

        # Lockable widgets during async actions
        # Exclude Tailscale from this list to keep it independent
        self.lockable_widgets = [
            self.cb_dryrun, self.btn_env, self.dd_rasterizer,
            self.btn_deps, self.dd_datasource, self.txt_source_path, self.btn_browse,
            self.btn_data, self.cb_antialiasing, self.cb_exposure,
            self.cb_depth, self.txt_depth_path, self.btn_train, self.btn_view
        ]



In [ ]:
# @title 4.2. GUI Controller (Logic)
# ===========================
# Event binding and Logic
# ===========================
import os


class GUIController:
    def __init__(self, view):
        self.view = view
        self._bind_events()
        self.current_path = os.getcwd()
        self._update_browser()

    def _bind_events(self):
        # Tailscale
        if 'tailscale_conn' in globals():
            self.view.cb_tailscale.observe(self._on_tailscale_change, names='value')
            tailscale_conn.status_callback = lambda msg: setattr(self.view.lbl_tailscale_status, 'value', msg)
        
        # Visibility Logic
        self.view.dd_rasterizer.observe(self._on_rasterizer_change, names='value')
        self.view.cb_depth.observe(self._on_depth_change, names='value')
        self.view.dd_datasource.observe(self._on_datasource_change, names='value')
        
        # File Browser
        self.view.btn_browse.on_click(self._toggle_browser)
        self.view.btn_up.on_click(self._on_up)
        self.view.sel_files.observe(self._on_select_item, names='value')
        self.view.btn_select.on_click(self._on_confirm_select)
        
        # Actions with Async Feedback
        self.view.btn_env.on_click(lambda b: self._run_action("Setting up Env...", setup_env))
        self.view.btn_deps.on_click(lambda b: self._run_action("Installing Deps...", install_deps))
        self.view.btn_data.on_click(lambda b: self._run_action("Preparing Data...", prepare_data))
        self.view.btn_train.on_click(lambda b: self._run_action("Training...", start_training))
        # Viewer is also long running but usually non-blocking subprocess. 
        # However, start_viewer function blocks while waiting for tunnel.
        self.view.btn_view.on_click(lambda b: self._run_action("Starting Viewer...", start_viewer))

    def _run_action(self, msg, func):
        self.view.set_status('busy', msg)
        for w in self.view.lockable_widgets:
            w.disabled = True

        try:
            func(None)
        except Exception as e:
            log(f"Error: {e}")
        finally:
            self.view.set_status('ready', "")
            for w in self.view.lockable_widgets:
                w.disabled = False

    # --- Event Handlers ---
    def _on_rasterizer_change(self, change):
        if change['new'].startswith('Accelerated'):
            self.view.cb_sparse_adam.value = True
        else:
            self.view.cb_sparse_adam.value = False

    def _on_depth_change(self, change):
        if change['new']:
            self.view.txt_depth_path.layout.display = 'flex'
        else:
            self.view.txt_depth_path.layout.display = 'none'

    def _on_datasource_change(self, change):
        val = change['new']
        if val in ['Google Drive', 'Local Folder']:
            self.view.txt_source_path.layout.display = 'flex'
            self.view.btn_browse.layout.display = 'block'
            self.view.lbl_upload_instruction.layout.display = 'none'
        elif val == 'Upload Zip':
            self.view.txt_source_path.layout.display = 'none'
            self.view.btn_browse.layout.display = 'none'
            self.view.file_upload.layout.display = 'block'
            self.view.lbl_upload_instruction.layout.display = 'block'
        elif val == 'Custom URL':
            self.view.txt_source_path.layout.display = 'flex'
            self.view.btn_browse.layout.display = 'none'
            self.view.file_upload.layout.display = 'none'
            self.view.lbl_upload_instruction.layout.display = 'none'
        else: # Demo Data
            self.view.txt_source_path.layout.display = 'none'
            self.view.btn_browse.layout.display = 'none'
            self.view.file_upload.layout.display = 'none'
            self.view.lbl_upload_instruction.layout.display = 'none'

    # --- Browser Logic ---
    def _update_browser(self):
        try:
            if not os.path.exists(self.current_path): self.current_path = '/content'
            items = sorted(os.listdir(self.current_path))
            formatted_items = []
            for item in items:
                if os.path.isdir(os.path.join(self.current_path, item)):
                    formatted_items.append(f"📁 {item}")
                else:
                    formatted_items.append(f"📄 {item}")
            self.view.sel_files.options = formatted_items
            self.view.lbl_path.value = f"Current: {self.current_path}"
        except Exception as e:
            self.view.lbl_path.value = f"Error: {e}"

    def _on_up(self, b):
        self.current_path = os.path.dirname(self.current_path)
        self._update_browser()

    def _on_select_item(self, change):
        if change['new']:
            name = change['new'].split(' ', 1)[1]
            full_path = os.path.join(self.current_path, name)
            if os.path.isdir(full_path):
                 self.current_path = full_path
                 self._update_browser()

    def _on_confirm_select(self, b):
        val = self.view.sel_files.value
        path_to_use = self.current_path
        if val:
            name = val.split(' ', 1)[1]
            path_to_use = os.path.join(self.current_path, name)
        self.view.txt_source_path.value = path_to_use
        self.view.browser_box.layout.display = 'none'

    def _toggle_browser(self, b):
        if self.view.browser_box.layout.display == 'none':
            self.view.browser_box.layout.display = 'block'
            self._update_browser()
        else:
            self.view.browser_box.layout.display = 'none'


    def _on_tailscale_change(self, change):
        should_connect = change['new']
        self.view.cb_tailscale.disabled = True
        try:
            if 'tailscale_conn' in globals():
                if should_connect:
                    tailscale_conn.connect()
                else:
                    tailscale_conn.disconnect()
        except Exception as e:
            log(f"Tailscale Error: {e}")
            if hasattr(self.view, 'lbl_tailscale_status'):
                 self.view.lbl_tailscale_status.value = "Error"
        finally:
            self.view.cb_tailscale.disabled = False


In [ ]:
# @title 5. Main Execution
if __name__ == "__main__":
    # 1. Instantiate View
    view = GUIWidgets()
    
    # 2. Restore logic from Cell 4.2 (Global Exports & Status)
    if 'tailscale_conn' in globals() and tailscale_conn.connected:
        view.cb_tailscale.value = True

    # Export widgets to globals for action functions
    globals()['file_upload_widget'] = view.file_upload
    globals()['cb_tailscale'] = view.cb_tailscale
    globals()['cb_dryrun'] = view.cb_dryrun
    globals()['dd_datasource'] = view.dd_datasource
    globals()['txt_source_path'] = view.txt_source_path
    globals()['dd_rasterizer'] = view.dd_rasterizer
    globals()['cb_antialiasing'] = view.cb_antialiasing
    globals()['cb_exposure'] = view.cb_exposure
    globals()['cb_depth'] = view.cb_depth
    globals()['txt_depth_path'] = view.txt_depth_path
    globals()['cb_sparse_adam'] = view.cb_sparse_adam
    
    # 3. Instantiate Controller
    controller = GUIController(view)
    
    # 4. Initialize State
    controller._on_datasource_change({'new': view.dd_datasource.value})
    
    # 5. Display GUI
    display(view.container)
